# ControlGuard: train A1 and A2 on Colab Pro
Select L4 or A100 for 8B. Select A100 40 GB for 32B. The notebook writes adapter weights and metrics to Google Drive.
The notebook supplies the project source and can generate synthetic data. To use pre-generated data, upload the JSONL files and enter their directory. Do not upload private company or employee data. The notebook uses no Gemini API.


In [ ]:
# @title Training settings
base = "Qwen3-8B" # @param ["Qwen3-8B", "Qwen3-32B"]
adapter = "both" # @param ["A1", "A2", "both"]
steps = 40 # @param {type:"integer"}
eval_examples = 8 # @param {type:"integer"}
dataset_folder = "" # @param {type:"string"}
project_folder = "" # @param {type:"string"}
drive_folder = "ControlGuard-Adapters" # @param {type:"string"}


In [ ]:
# Unpack the source bundle that this repository's packager supplies.
from base64 import b64decode
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

runtime_root = Path(globals().get("RUNTIME_ROOT", "/content/controlguard-training"))
runtime_root.mkdir(parents=True, exist_ok=True)
bundle_text = (
    "..."
    "Mi5qc29uUEsFBgAAAAAbABsAbQcAAKOTAAAAAA=="
)
if project_folder.strip():
    source = Path(project_folder).expanduser().resolve()
    if not (source / "controlguard/llm/train.py").is_file() or not (source / "llm-config").is_dir():
        raise FileNotFoundError("Use the root of an uploaded or cloned ControlGuard checkout.")
else:
    source = runtime_root / "source"
    source.mkdir(exist_ok=True)
    with ZipFile(BytesIO(b64decode(bundle_text))) as archive:
        archive.extractall(source)
print("ControlGuard source ready.")


In [ ]:
import os
import subprocess
import sys

if base not in ("Qwen3-8B", "Qwen3-32B") or adapter not in ("A1", "A2", "both"):
    raise ValueError("Choose a listed base and adapter.")
if not 1 <= steps <= 250 or not 1 <= eval_examples <= 32:
    raise ValueError("Use 1–250 steps and 1–32 evaluation examples.")
try:
    report = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
        text=True, capture_output=True, check=True,
    ).stdout
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError("Select a GPU runtime before running the training cells.") from error
minimum_mib = 39000 if base == "Qwen3-32B" else 14000
selected = None
for line in report.strip().splitlines():
    index, name, memory, capability = (field.strip() for field in line.split(","))
    if int(memory) >= minimum_mib and float(capability) >= (8.0 if base == "Qwen3-32B" else 7.0):
        selected = (index, name, memory)
        break
if selected is None:
    raise RuntimeError(f"{base} needs a supported GPU with at least {minimum_mib} MiB. Use T4/L4 for 8B or A100 40 GB for 32B.")
env = {**os.environ, "CUDA_VISIBLE_DEVICES": selected[0], "TOKENIZERS_PARALLELISM": "false"}
print(f"Using one {selected[1]} with {selected[2]} MiB.")


In [ ]:
from google.colab import drive

folder = Path(drive_folder)
if not drive_folder.strip() or folder.is_absolute() or ".." in folder.parts:
    raise ValueError("Use a folder name within My Drive.")
drive.mount("/content/drive")
output = Path("/content/drive/MyDrive") / folder


In [ ]:
packages = [
    "unsloth==2026.11.6", "unsloth_zoo==2026.11.6", "transformers==4.56.2",
    "trl==0.23.1", "peft==0.18.0", "datasets==4.3.0", "bitsandbytes==0.47.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


In [ ]:
if dataset_folder.strip():
    data = Path(dataset_folder).expanduser().resolve()
else:
    data = runtime_root / "data"
    subprocess.run([
        sys.executable, "-m", "controlguard.llm.dataset", "--output", str(data),
        "--train-count", "512", "--eval-count", "32",
    ], cwd=source, check=True)
adapters = ("a1", "a2") if adapter == "both" else (adapter.lower(),)
for family in adapters:
    for split in ("train", "eval"):
        if not (data / f"{family}-{split}.jsonl").is_file():
            raise FileNotFoundError(f"Missing {family}-{split}.jsonl in {data}.")


In [ ]:
import json

output.mkdir(parents=True, exist_ok=True)
for family in adapters:
    config_path = f"kaggle-{family}" if base == "Qwen3-8B" else f"qwen32b-{family}"
    config = json.loads((source / "llm-config" / f"{config_path}.json").read_text())
    config.update(mode="unsloth-qlora", steps=steps, eval_limit=eval_examples, max_length=1024)
    job_config = runtime_root / f"{family}-resolved.json"
    target = output / base / family
    job_config.write_text(json.dumps(config, allow_nan=False))
    subprocess.run([
        sys.executable, "-m", "controlguard.llm.train", str(job_config),
        "--data", str(data), "--output", str(target),
    ], cwd=source, env=env, check=True)
    weights = target / "adapter" / "adapter_model.safetensors"
    if not weights.is_file():
        raise FileNotFoundError(f"No saved safetensors adapter at {weights}.")
    print(f"{family.upper()} weights: {weights}")
    print((target / "metrics.json").read_text())
